In [1]:
from openai import OpenAI
import fitz
import faiss
import pandas

In [ ]:
from pathlib import Path
import pymupdf4llm

PDF_FOLDER = Path("data/pdf")
MD_FOLDER = Path("data/markdown")

MD_FOLDER.mkdir(parents=True, exist_ok=True)

pdf_files = list(PDF_FOLDER.glob("*.pdf"))

print(f"{len(pdf_files)} PDF found")

# for pdf_path in pdf_files:

    md_filename = pdf_path.stem + ".md"
    output_md_path = MD_FOLDER / md_filename

    print(f"Converting: {pdf_path.name}")

    md_text = pymupdf4llm.to_markdown(str(pdf_path))

    output_md_path.write_text(md_text, encoding="utf-8")

print("Conversion finished")

14 PDF found
Converting: PI_COM_C(2024)902_EN_TXT.pdf
Converting: OJ_L_202401772_EN_TXT.pdf
Converting: OJ_L_202500420_EN_TXT.pdf
Converting: OJ_L_202500301_EN_TXT.pdf
Converting: CELEX_32022R2554_EN_TXT.pdf
Converting: OJ_L_202500302_EN_TXT.pdf
Converting: PI_COM_C(2024)896_EN_TXT.pdf
Converting: OJ_L_202401773_EN_TXT.pdf
Converting: OJ_L_202401774_EN_TXT.pdf
Converting: OJ_L_202402956_EN_TXT.pdf
Converting: CELEX_32022L2556_FR_TXT.pdf
Converting: OJ_L_202500295_EN_TXT.pdf
Converting: OJ_L_202501190_EN_TXT.pdf
Converting: OJ_L_202500532_EN_TXT.pdf
Conversion finished


In [3]:
import re

PATTERNS = {
    # document
    "doc_title": re.compile(r"^#\s+\*\*(.+?)\*\*\s*$"),
    "doc_date": re.compile(r"^##\s+\*\*((?:of|du).+?)\*\*\s*$", re.IGNORECASE),
    "bold_line": re.compile(r"^\*\*(.+?)\*\*\s*$"),

    # hiérarchie
    "title": re.compile(r"^#\s+TITLE\s+(.+?)\s*$", re.IGNORECASE),
    "chapter": re.compile(r"^#\s+_?CHAPTER\s+(.+?)_?\s*$", re.IGNORECASE),
    "section": re.compile(r"^##\s+Section\s+(.+?)\s*$", re.IGNORECASE),
    "article": re.compile(r"^##\s+_?Article\s+(.+?)_?\s*$", re.IGNORECASE),
}

In [4]:
def clean_number(text):
    if not text:
        return None

    # supprime markdown et caractères parasites
    text = re.sub(r"[_*\-`]", "", text)

    # garde uniquement lettres + chiffres (ex: II, 2, 15a)
    text = re.sub(r"[^0-9A-Za-z]", "", text)

    text = text.strip()

    return text if text else None


def normalize_date(text):
    if not text:
        return None
    text = re.sub(r"^(of|du)\s+", "", text, flags=re.IGNORECASE)
    return text.strip()



In [5]:
def find_next_title(lines, start_idx):

    for j in range(start_idx + 1, len(lines)):
        l = lines[j].strip()

        if not l:
            continue

        # stop si on rencontre un autre bloc structurel
        if any(p.match(l) for p in [
            PATTERNS["title"],
            PATTERNS["chapter"],
            PATTERNS["section"],
            PATTERNS["article"]
        ]):
            return None

        # match strict : ## **...**
        m = re.match(r"^##\s+\*\*(.+?)\*\*\s*$", l)
        if m:
            title = m.group(1).strip()

            # sécurité : éviter les faux titres trop longs
            if 3 <= len(title) <= 300:
                return title

    return None

In [6]:
from copy import deepcopy

def clean_text_for_block(text):
    lines = text.splitlines()
    cleaned = []

    for line in lines:
        l = line.strip()

        if not l:
            cleaned.append("")
            continue

        # retire séparateurs markdown purs
        if re.fullmatch(r"[-_]{3,}", l):
            continue

        cleaned.append(line)

    text = "\n".join(cleaned)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def parse_document(md_text, doc_id):
    lines = md_text.splitlines()

    metadata_global = {
        "doc_id": doc_id,
        "doc_title": None,
        "doc_date": None,
        "doc_subtitle": None,
    }

    current = {
        "title_nb": None,
        "title_name": None,
        "chapter_nb": None,
        "chapter_name": None,
        "section_nb": None,
        "section_name": None,
        "article_nb": None,
        "article_name": None,
    }

    blocks = []
    current_block = None
    intro_lines = []
    intro_started = False
    intro_closed = False
    block_id = 0

    i = 0

    while i < len(lines):
        line = lines[i].strip()

        if not line:
            if current_block is not None:
                current_block["content"].append("")
            elif intro_started and not intro_closed:
                intro_lines.append("")
            i += 1
            continue

        # ---------------- DOC TITLE ----------------
        m = PATTERNS["doc_title"].match(line)
        if m and metadata_global["doc_title"] is None:
            metadata_global["doc_title"] = m.group(1).strip()
            intro_started = True
            i += 1
            continue

        # ---------------- DOC DATE ----------------
        m = PATTERNS["doc_date"].match(line)
        if m and metadata_global["doc_date"] is None:
            metadata_global["doc_date"] = normalize_date(m.group(1))
            i += 1

            # premier **...** après la date = sous-titre
            for j in range(i, len(lines)):
                l2 = lines[j].strip()
                if not l2:
                    continue

                if any(p.match(l2) for p in [
                    PATTERNS["title"],
                    PATTERNS["chapter"],
                    PATTERNS["section"],
                    PATTERNS["article"]
                ]):
                    break

                m_sub = PATTERNS["bold_line"].match(l2)
                if m_sub:
                    metadata_global["doc_subtitle"] = m_sub.group(1).strip()
                    break

            continue

        # ---------------- TITLE ----------------
        m = PATTERNS["title"].match(line)
        if m:
            if not intro_closed:
                intro_closed = True
                intro_text = clean_text_for_block("\n".join(intro_lines))
                if intro_text:
                    blocks.append({
                        **deepcopy(metadata_global),
                        "title_nb": None,
                        "title_name": None,
                        "chapter_nb": None,
                        "chapter_name": None,
                        "section_nb": None,
                        "section_name": None,
                        "article_nb": None,
                        "article_name": None,
                        "block_id": block_id,
                        "block_type": "introduction",
                        "text": intro_text
                    })
                    block_id += 1

            current["title_nb"] = clean_number(m.group(1))
            current["title_name"] = find_next_title(lines, i)
            i += 1
            continue

        # ---------------- CHAPTER ----------------
        m = PATTERNS["chapter"].match(line)
        if m:
            if not intro_closed:
                intro_closed = True
                intro_text = clean_text_for_block("\n".join(intro_lines))
                if intro_text:
                    blocks.append({
                        **deepcopy(metadata_global),
                        "title_nb": None,
                        "title_name": None,
                        "chapter_nb": None,
                        "chapter_name": None,
                        "section_nb": None,
                        "section_name": None,
                        "article_nb": None,
                        "article_name": None,
                        "block_id": block_id,
                        "block_type": "introduction",
                        "text": intro_text
                    })
                    block_id += 1

            current["chapter_nb"] = clean_number(m.group(1))
            current["chapter_name"] = find_next_title(lines, i)
            i += 1
            continue

        # ---------------- SECTION ----------------
        m = PATTERNS["section"].match(line)
        if m:
            if not intro_closed:
                intro_closed = True
                intro_text = clean_text_for_block("\n".join(intro_lines))
                if intro_text:
                    blocks.append({
                        **deepcopy(metadata_global),
                        "title_nb": None,
                        "title_name": None,
                        "chapter_nb": None,
                        "chapter_name": None,
                        "section_nb": None,
                        "section_name": None,
                        "article_nb": None,
                        "article_name": None,
                        "block_id": block_id,
                        "block_type": "introduction",
                        "text": intro_text
                    })
                    block_id += 1

            current["section_nb"] = clean_number(m.group(1))
            current["section_name"] = find_next_title(lines, i)
            i += 1
            continue

        # ---------------- ARTICLE ----------------
        m = PATTERNS["article"].match(line)
        if m:
            if not intro_closed:
                intro_closed = True
                intro_text = clean_text_for_block("\n".join(intro_lines))
                if intro_text:
                    blocks.append({
                        **deepcopy(metadata_global),
                        "title_nb": None,
                        "title_name": None,
                        "chapter_nb": None,
                        "chapter_name": None,
                        "section_nb": None,
                        "section_name": None,
                        "article_nb": None,
                        "article_name": None,
                        "block_id": block_id,
                        "block_type": "introduction",
                        "text": intro_text
                    })
                    block_id += 1

            if current_block:
                current_block["text"] = clean_text_for_block("\n".join(current_block["content"]))
                del current_block["content"]
                blocks.append(current_block)
                block_id += 1

            current["article_nb"] = clean_number(m.group(1))
            current["article_name"] = find_next_title(lines, i)

            current_block = {
                **deepcopy(metadata_global),
                **deepcopy(current),
                "block_id": block_id,
                "block_type": "article",
                "content": []
            }

            i += 1
            continue

        # ---------------- CONTENU ----------------
        if current_block:
            current_block["content"].append(line)
        elif intro_started and not intro_closed:
            intro_lines.append(line)

        i += 1

    # intro seule si aucun bloc structurel n'est venu la fermer
    if intro_started and not intro_closed:
        intro_text = clean_text_for_block("\n".join(intro_lines))
        if intro_text:
            blocks.append({
                **deepcopy(metadata_global),
                "title_nb": None,
                "title_name": None,
                "chapter_nb": None,
                "chapter_name": None,
                "section_nb": None,
                "section_name": None,
                "article_nb": None,
                "article_name": None,
                "block_id": block_id,
                "block_type": "introduction",
                "text": intro_text
            })
            block_id += 1

    # dernier article
    if current_block:
        current_block["text"] = clean_text_for_block("\n".join(current_block["content"]))
        del current_block["content"]
        blocks.append(current_block)

    return blocks

In [7]:
def extract_intro_block(md_text, doc_id):

    lines = md_text.splitlines()

    intro_lines = []
    started = False

    for line in lines:

        if any(p.match(line.strip()) for p in [
            PATTERNS["title"],
            PATTERNS["chapter"],
            PATTERNS["section"],
            PATTERNS["article"]
        ]):
            break

        if "**" in line or "#" in line:
            started = True
            continue

        if started:
            intro_lines.append(line)

    return {
        "doc_id": doc_id,
        "block_type": "introduction",
        "text": "\n".join(intro_lines).strip()
    }

In [8]:
from pathlib import Path

md_path = Path("data/markdown/CELEX_32022L2556_FR_TXT.md")

md_text = md_path.read_text(encoding="utf-8", errors="ignore")

blocks = parse_document(md_text, doc_id=md_path.stem)

print(f"Total blocks: {len(blocks)}")
print("\nSample blocks:\n")

for b in blocks[:3]:
    print("=" * 100)
    print("BLOCK_TYPE :", b["block_type"])
    print("DOC        :", b["doc_title"])
    print("DOC_DATE   :", b["doc_date"])
    print("SUB_DOC    :", b["doc_subtitle"])
    print("TITLE      :", b["title_nb"], b["title_name"])
    print("CHAPTER    :", b["chapter_nb"], b["chapter_name"])
    print("SECTION    :", b["section_nb"], b["section_name"])
    print("ARTICLE    :", b["article_nb"], b["article_name"])
    print("---")
    print(b["text"])
    print()

Total blocks: 12

Sample blocks:

BLOCK_TYPE : introduction
DOC        : DIRECTIVE (UE) 2022/2556 DU PARLEMENT EUROPÉEN ET DU CONSEIL
DOC_DATE   : 14 décembre 2022
SUB_DOC    : modifiant les directives 2009/65/CE, 2009/138/CE, 2011/61/UE, 2013/36/UE, 2014/59/UE, 2014/65/UE, (UE) 2015/2366 et (UE) 2016/2341 en ce qui concerne la résilience opérationnelle numérique du secteur financier
TITLE      : None None
CHAPTER    : None None
SECTION    : None None
ARTICLE    : None None
---
**modifiant les directives 2009/65/CE, 2009/138/CE, 2011/61/UE, 2013/36/UE, 2014/59/UE, 2014/65/UE, (UE) 2015/2366 et (UE) 2016/2341 en ce qui concerne la résilience opérationnelle numérique du secteur financier**

LE PARLEMENT EUROPÉEN ET LE CONSEIL DE L’UNION EUROPÉENNE,

vu le traité sur le fonctionnement de l’Union européenne, et notamment son article 53, paragraphe 1, et son article 114,

vu la proposition de la Commission européenne,

après transmission du projet d’acte législatif aux parlements nationaux,

In [11]:
intro

{'doc_id': 'CELEX_32022L2556_FR_TXT',
 'block_type': 'introduction',
 'text': '(Texte présentant de l’intérêt pour l’EEE)\n\nLE PARLEMENT EUROPÉEN ET LE CONSEIL DE L’UNION EUROPÉENNE, \n\nvu le traité sur le fonctionnement de l’Union européenne, et notamment son article 53, paragraphe 1, et son article 114, \n\nvu la proposition de la Commission européenne, \n\naprès transmission du projet d’acte législatif aux parlements nationaux, \n\nvu l’avis de la Banque centrale européenne ([1] ), \n\nvu l’avis du Comité économique et social européen ([2] ), \n\nstatuant conformément à la procédure législative ordinaire ([3] ), \n\nconsidérant ce qui suit: \n\n- (1) L’Union doit traiter de manière adéquate et globale les risques numériques auxquels sont exposées toutes les entités financières et qui découlent d’un recours accru aux technologies de l’information et de la communication (TIC) dans le cadre de la fourniture et de la consommation de services financiers, ce qui contribuera à exploiter 

In [12]:
blocks[1]

{'doc_id': 'CELEX_32022L2556_FR_TXT',
 'doc_title': 'DIRECTIVE (UE) 2022/2556 DU PARLEMENT EUROPÉEN ET DU CONSEIL',
 'doc_date': '14 décembre 2022',
 'doc_subtitle': 'modifiant les directives 2009/65/CE, 2009/138/CE, 2011/61/UE, 2013/36/UE, 2014/59/UE, 2014/65/UE, (UE) 2015/2366 et (UE) 2016/2341 en ce qui concerne la résilience opérationnelle numérique du secteur financier',
 'title_nb': None,
 'title_name': None,
 'chapter_nb': None,
 'chapter_name': None,
 'section_nb': None,
 'section_name': None,
 'article_nb': '1',
 'article_name': 'Modifications de la directive 2009/65/CE',
 'block_id': 1,
 'block_type': 'article',
 'text': '## **Modifications de la directive 2009/65/CE**\n\nL’article 12 de la directive 2009/65/CE est modifié comme suit:\n\n- 1) Au paragraphe 1, deuxième alinéa, le point a) est remplacé par le texte suivant:\n\n- «a) ait des procédures administratives et comptables saines, des dispositifs de contrôle et de sauvegarde dans le domaine du traitement électronique de

In [13]:
from pathlib import Path

MD_FOLDER = Path("data/markdown")

all_blocks = []

md_files = sorted(MD_FOLDER.glob("*.md"))

print(f"{len(md_files)} markdown files found\n")

for md_path in md_files:
    
    print(f"Processing: {md_path.name}")
    
    md_text = md_path.read_text(encoding="utf-8", errors="ignore")
    
    blocks = parse_document(md_text, doc_id=md_path.stem)
    
    print(f" → {len(blocks)} blocks\n")
    
    all_blocks.extend(blocks)

print("=" * 100)
print(f"TOTAL BLOCKS: {len(all_blocks)}")

14 markdown files found

Processing: CELEX_32022L2556_FR_TXT.md
 → 12 blocks

Processing: CELEX_32022R2554_EN_TXT.md
 → 65 blocks

Processing: OJ_L_202401772_EN_TXT.md
 → 14 blocks

Processing: OJ_L_202401773_EN_TXT.md
 → 12 blocks

Processing: OJ_L_202401774_EN_TXT.md
 → 43 blocks

Processing: OJ_L_202402956_EN_TXT.md
 → 8 blocks

Processing: OJ_L_202500295_EN_TXT.md
 → 8 blocks

Processing: OJ_L_202500301_EN_TXT.md
 → 8 blocks

Processing: OJ_L_202500302_EN_TXT.md
 → 10 blocks

Processing: OJ_L_202500420_EN_TXT.md
 → 7 blocks

Processing: OJ_L_202500532_EN_TXT.md
 → 8 blocks

Processing: OJ_L_202501190_EN_TXT.md
 → 18 blocks

Processing: PI_COM_C(2024)896_EN_TXT.md
 → 8 blocks

Processing: PI_COM_C(2024)902_EN_TXT.md
 → 8 blocks

TOTAL BLOCKS: 229


In [16]:
all_blocks[1]

{'doc_id': 'CELEX_32022L2556_FR_TXT',
 'doc_title': 'DIRECTIVE (UE) 2022/2556 DU PARLEMENT EUROPÉEN ET DU CONSEIL',
 'doc_date': '14 décembre 2022',
 'doc_subtitle': 'modifiant les directives 2009/65/CE, 2009/138/CE, 2011/61/UE, 2013/36/UE, 2014/59/UE, 2014/65/UE, (UE) 2015/2366 et (UE) 2016/2341 en ce qui concerne la résilience opérationnelle numérique du secteur financier',
 'title_nb': None,
 'title_name': None,
 'chapter_nb': None,
 'chapter_name': None,
 'section_nb': None,
 'section_name': None,
 'article_nb': '1',
 'article_name': 'Modifications de la directive 2009/65/CE',
 'block_id': 1,
 'block_type': 'article',
 'text': '## **Modifications de la directive 2009/65/CE**\n\nL’article 12 de la directive 2009/65/CE est modifié comme suit:\n\n- 1) Au paragraphe 1, deuxième alinéa, le point a) est remplacé par le texte suivant:\n\n- «a) ait des procédures administratives et comptables saines, des dispositifs de contrôle et de sauvegarde dans le domaine du traitement électronique de

Parametre de chunking

In [24]:
import re
import math
import numpy as np
import pandas as pd
import faiss

from pathlib import Path
from openai import OpenAI

from Env import OPENAI_API_KEY


client = OpenAI(api_key=OPENAI_API_KEY)

VECTOR_STORE_DIR = Path("vector_store")
VECTOR_STORE_DIR.mkdir(parents=True, exist_ok=True)

EMBEDDING_MODEL = "text-embedding-3-small"

MAX_CHUNK_CHARS = 2500
OVERLAP_CHARS = 350
MIN_CHUNK_CHARS = 500
BATCH_SIZE = 100

cleaning des bloc de texte ---____===*** etc pour n epas impacté la sémantique des chunk lors de la vectorisation 

In [18]:
def clean_text_for_embedding(text: str) -> str:
    if not text:
        return ""

    text = text.replace("\u00a0", " ")
    text = text.replace("\xad", "")
    
    # retire séparateurs markdown purs
    text = re.sub(r"(?m)^[ \t]*[-_]{3,}[ \t]*$", "", text)

    # retire heading markdown en début de ligne, mais garde le contenu
    text = re.sub(r"(?m)^#{1,6}\s+", "", text)

    # retire emphase markdown
    text = text.replace("**", "")
    text = text.replace("__", "")
    text = text.replace("*", "")
    text = text.replace("_", "")

    # espaces
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()

In [19]:
for b in all_blocks:
    b["text_raw"] = b["text"]
    b["text_for_embedding"] = clean_text_for_embedding(b["text"])

Découpage des bloc en sous chunk :

In [20]:
def split_text_with_overlap(text, max_chars=MAX_CHUNK_CHARS, overlap_chars=OVERLAP_CHARS):
    text = text.strip()
    if not text:
        return []

    if len(text) <= max_chars:
        return [text]

    paragraphs = [p.strip() for p in text.split("\n\n") if p.strip()]
    if not paragraphs:
        paragraphs = [text]

    chunks = []
    current = ""

    for para in paragraphs:
        candidate = (current + "\n\n" + para).strip() if current else para

        if len(candidate) <= max_chars:
            current = candidate
        else:
            if current:
                chunks.append(current)

            if len(para) <= max_chars:
                current = para
            else:
                # fallback si un seul paragraphe est trop long
                start = 0
                while start < len(para):
                    end = start + max_chars
                    piece = para[start:end].strip()
                    if piece:
                        chunks.append(piece)
                    start += max_chars - overlap_chars
                current = ""

    if current:
        chunks.append(current)

    # petit merge des chunks trop petits
    merged = []
    buffer = ""

    for ch in chunks:
        if not buffer:
            buffer = ch
        elif len(buffer) < MIN_CHUNK_CHARS and len(buffer) + 2 + len(ch) <= max_chars:
            buffer = buffer + "\n\n" + ch
        else:
            merged.append(buffer)
            buffer = ch

    if buffer:
        merged.append(buffer)

    return merged

création des chunks + chunks id 

In [21]:
def build_chunks_from_blocks(blocks):
    chunks = []

    for block in blocks:
        block_text = block.get("text_for_embedding", "").strip()
        if not block_text:
            continue

        sub_chunks = split_text_with_overlap(block_text)
        subchunk_count = len(sub_chunks)

        for chunk_nb, chunk_text in enumerate(sub_chunks):
            chunk_id = f"{block['doc_id']}__block_{block['block_id']}__chunk_{chunk_nb}"

            chunk = {
                "chunk_id": chunk_id,
                "doc_id": block["doc_id"],
                "block_id": block["block_id"],
                "chunk_nb": chunk_nb,
                "subchunk_count": subchunk_count,

                "doc_title": block["doc_title"],
                "doc_date": block["doc_date"],
                "doc_subtitle": block["doc_subtitle"],

                "title_nb": block["title_nb"],
                "title_name": block["title_name"],
                "chapter_nb": block["chapter_nb"],
                "chapter_name": block["chapter_name"],
                "section_nb": block["section_nb"],
                "section_name": block["section_name"],
                "article_nb": block["article_nb"],
                "article_name": block["article_name"],

                "block_type": block["block_type"],

                "text_raw": block["text_raw"],
                "text_for_embedding": chunk_text,
                "char_count": len(chunk_text),
            }

            chunks.append(chunk)

    # voisins internes à la liste complète
    for i, ch in enumerate(chunks):
        ch["prev_chunk_id"] = chunks[i - 1]["chunk_id"] if i > 0 else None
        ch["next_chunk_id"] = chunks[i + 1]["chunk_id"] if i < len(chunks) - 1 else None

    return chunks

construire les chunks 

In [22]:
all_chunks = build_chunks_from_blocks(all_blocks)

print(f"Total chunks: {len(all_chunks)}")
all_chunks[:2]

Total chunks: 472


[{'chunk_id': 'CELEX_32022L2556_FR_TXT__block_0__chunk_0',
  'doc_id': 'CELEX_32022L2556_FR_TXT',
  'block_id': 0,
  'chunk_nb': 0,
  'subchunk_count': 7,
  'doc_title': 'DIRECTIVE (UE) 2022/2556 DU PARLEMENT EUROPÉEN ET DU CONSEIL',
  'doc_date': '14 décembre 2022',
  'doc_subtitle': 'modifiant les directives 2009/65/CE, 2009/138/CE, 2011/61/UE, 2013/36/UE, 2014/59/UE, 2014/65/UE, (UE) 2015/2366 et (UE) 2016/2341 en ce qui concerne la résilience opérationnelle numérique du secteur financier',
  'title_nb': None,
  'title_name': None,
  'chapter_nb': None,
  'chapter_name': None,
  'section_nb': None,
  'section_name': None,
  'article_nb': None,
  'article_name': None,
  'block_type': 'introduction',
  'text_raw': '**modifiant les directives 2009/65/CE, 2009/138/CE, 2011/61/UE, 2013/36/UE, 2014/59/UE, 2014/65/UE, (UE) 2015/2366 et (UE) 2016/2341 en ce qui concerne la résilience opérationnelle numérique du secteur financier**\n\nLE PARLEMENT EUROPÉEN ET LE CONSEIL DE L’UNION EUROPÉENNE

exploration validation des chunks : 

In [23]:
for ch in all_chunks[:3]:
    print("=" * 120)
    print("CHUNK_ID    :", ch["chunk_id"])
    print("DOC         :", ch["doc_title"])
    print("DOC_DATE    :", ch["doc_date"])
    print("SUB_DOC     :", ch["doc_subtitle"])
    print("BLOCK_TYPE  :", ch["block_type"])
    print("TITLE       :", ch["title_nb"], ch["title_name"])
    print("CHAPTER     :", ch["chapter_nb"], ch["chapter_name"])
    print("SECTION     :", ch["section_nb"], ch["section_name"])
    print("ARTICLE     :", ch["article_nb"], ch["article_name"])
    print("CHUNK_NB    :", ch["chunk_nb"], "/", ch["subchunk_count"] - 1)
    print("CHAR_COUNT  :", ch["char_count"])
    print("---")
    print(ch["text_for_embedding"])
    print()

CHUNK_ID    : CELEX_32022L2556_FR_TXT__block_0__chunk_0
DOC         : DIRECTIVE (UE) 2022/2556 DU PARLEMENT EUROPÉEN ET DU CONSEIL
DOC_DATE    : 14 décembre 2022
SUB_DOC     : modifiant les directives 2009/65/CE, 2009/138/CE, 2011/61/UE, 2013/36/UE, 2014/59/UE, 2014/65/UE, (UE) 2015/2366 et (UE) 2016/2341 en ce qui concerne la résilience opérationnelle numérique du secteur financier
BLOCK_TYPE  : introduction
TITLE       : None None
CHAPTER     : None None
SECTION     : None None
ARTICLE     : None None
CHUNK_NB    : 0 / 6
CHAR_COUNT  : 2167
---
modifiant les directives 2009/65/CE, 2009/138/CE, 2011/61/UE, 2013/36/UE, 2014/59/UE, 2014/65/UE, (UE) 2015/2366 et (UE) 2016/2341 en ce qui concerne la résilience opérationnelle numérique du secteur financier

LE PARLEMENT EUROPÉEN ET LE CONSEIL DE L’UNION EUROPÉENNE,

vu le traité sur le fonctionnement de l’Union européenne, et notamment son article 53, paragraphe 1, et son article 114,

vu la proposition de la Commission européenne,

après t

creation des meta data liée et vector store par chunkID 

In [25]:
chunks_df = pd.DataFrame(all_chunks)

metadata_path = VECTOR_STORE_DIR / "metadata.parquet"
chunks_df.to_parquet(metadata_path, index=False)

print(f"Saved metadata: {metadata_path}")

Saved metadata: vector_store/metadata.parquet


In [26]:
import pandas as pd
import pyarrow
print(pd.__version__, pyarrow.__version__)

2.2.2 15.0.2


In [27]:
def embed_texts(texts, model=EMBEDDING_MODEL):
    response = client.embeddings.create(
        model=model,
        input=texts
    )
    return [item.embedding for item in response.data]

vectorisation des chunks 

In [28]:
texts = chunks_df["text_for_embedding"].tolist()

all_embeddings = []

for i in range(0, len(texts), BATCH_SIZE):
    batch = texts[i:i+BATCH_SIZE]
    batch_embeddings = embed_texts(batch)
    all_embeddings.extend(batch_embeddings)
    print(f"Embedded {min(i + BATCH_SIZE, len(texts))}/{len(texts)}")

embeddings_array = np.array(all_embeddings, dtype="float32")
print("Embeddings shape:", embeddings_array.shape)

Embedded 100/472
Embedded 200/472
Embedded 300/472
Embedded 400/472
Embedded 472/472
Embeddings shape: (472, 1536)


création du vectore store 

In [29]:
faiss.normalize_L2(embeddings_array)

dimension = embeddings_array.shape[1]
index = faiss.IndexFlatIP(dimension)
index.add(embeddings_array)

index_path = VECTOR_STORE_DIR / "faiss.index"
faiss.write_index(index, str(index_path))

print(f"Saved FAISS index: {index_path}")
print(f"Vectors in index: {index.ntotal}")

Saved FAISS index: vector_store/faiss.index
Vectors in index: 472


verification 

In [30]:
print("Metadata rows :", len(chunks_df))
print("FAISS vectors :", index.ntotal)
print("Same count    :", len(chunks_df) == index.ntotal)

print("\nExample chunk_id:")
print(chunks_df.iloc[0]["chunk_id"])

Metadata rows : 472
FAISS vectors : 472
Same count    : True

Example chunk_id:
CELEX_32022L2556_FR_TXT__block_0__chunk_0


In [48]:
chunks_df[chunks_df["chunk_id"] == "CELEX_32022R2554_EN_TXT__block_16__chunk_1"]

,chunk_id,doc_id,block_id,chunk_nb,subchunk_count,doc_title,doc_date,doc_subtitle,title_nb,title_name,...,section_nb,section_name,article_nb,article_name,block_type,text_raw,text_for_embedding,char_count,prev_chunk_id,next_chunk_id
108,CELEX_32022R2554_EN_TXT__block_16__chunk_1,CELEX_32022R2554_EN_TXT,16,1,2,REGULATION (EU) 2022/2554 OF THE EUROPEAN PARL...,14 December 2022,on digital operational resilience for the fina...,None,None,...,II,None,16,Simplified ICT risk management framework,article,## **Simplified ICT risk management framework*...,2. The ICT risk management framework referred ...,2447,CELEX_32022R2554_EN_TXT__block_16__chunk_0,CELEX_32022R2554_EN_TXT__block_17__chunk_0


In [31]:
from app.config import FAISS_INDEX_PATH, METADATA_PATH

print(FAISS_INDEX_PATH.exists())
print(METADATA_PATH.exists())

True
True


In [32]:
from app.openai_client import client

print(client is not None)

True


In [33]:
from app.openai_client import client
from app.config import EMBEDDING_MODEL

res = client.embeddings.create(
    model=EMBEDDING_MODEL,
    input="test"
)

print(len(res.data[0].embedding))

1536


In [34]:
from app.retrieval import retrieve, build_context

query = "What does Article 29 say about ICT concentration risk?"

results = retrieve(query)

print(f"Retrieved: {len(results)} chunks\n")

for r in results[:5]:
    print("=" * 100)
    print("CHUNK_ID   :", r.get("chunk_id"))
    print("DOC        :", r.get("doc_title"))
    print("ARTICLE    :", r.get("article_nb"), r.get("article_name"))
    print("CHAPTER    :", r.get("chapter_nb"), r.get("chapter_name"))
    print("SECTION    :", r.get("section_nb"), r.get("section_name"))
    print("---")
    print((r.get("text_for_embedding") or r.get("text", ""))[:1000])
    print()

context = build_context(results)
print(context[:3000])

Retrieved: 15 chunks

CHUNK_ID   : CELEX_32022R2554_EN_TXT__block_16__chunk_0
DOC        : REGULATION (EU) 2022/2554 OF THE EUROPEAN PARLIAMENT AND OF THE COUNCIL
ARTICLE    : 16 Simplified ICT risk management framework
CHAPTER    : II ICT risk management
SECTION    : II None
---
Simplified ICT risk management framework

1. Articles 5 to 15 of this Regulation shall not apply to small and non-interconnected investment firms, payment institutions exempted pursuant to Directive (EU) 2015/2366; institutions exempted pursuant to Directive 2013/36/EU in respect of which Member States have decided not to apply the option referred to in Article 2(4) of this Regulation; electronic money institutions exempted pursuant to Directive 2009/110/EC; and small institutions for occupational retirement provision.

Without prejudice to the first subparagraph, the entities listed in the first subparagraph shall:

- (a) put in place and maintain a sound and documented ICT risk management framework that deta

In [35]:
from app.retrieval import retrieve, build_context_payload, get_chunk_ids

query = "What does Article 29 say about ICT concentration risk in Title V?"

chunks = retrieve(query)
payload = build_context_payload(chunks)

print("Chunk IDs used:")
print(payload["chunk_ids"])

print("\nContext preview:\n")
print(payload["context"][:3000])

Chunk IDs used:
['CELEX_32022R2554_EN_TXT__block_16__chunk_0', 'CELEX_32022R2554_EN_TXT__block_16__chunk_1', 'CELEX_32022R2554_EN_TXT__block_28__chunk_0', 'CELEX_32022R2554_EN_TXT__block_28__chunk_1']

Context preview:

[chunk_id: CELEX_32022R2554_EN_TXT__block_16__chunk_0] [doc: REGULATION (EU) 2022/2554 OF THE EUROPEAN PARLIAMENT AND OF THE COUNCIL] [date: 14 December 2022] [title: - ] [chapter: II ICT risk management] [section: II ] [article: 16 Simplified ICT risk management framework]
Simplified ICT risk management framework

1. Articles 5 to 15 of this Regulation shall not apply to small and non-interconnected investment firms, payment institutions exempted pursuant to Directive (EU) 2015/2366; institutions exempted pursuant to Directive 2013/36/EU in respect of which Member States have decided not to apply the option referred to in Article 2(4) of this Regulation; electronic money institutions exempted pursuant to Directive 2009/110/EC; and small institutions for occupational re

In [45]:
from app.main import answer_question

res = answer_question("What does Article 29 say about ICT concentration risk?", return_context=True)

print(res["answer"])
print(res["chunk_ids"][:5])
print(res["sources"][:2])

The retrieved legal context does not provide the specific content of Article 29 regarding ICT concentration risk. It only mentions that financial entities must identify and assess all relevant risks in relation to contractual arrangements, including the possibility that such arrangements may contribute to reinforcing ICT concentration risk as referred to in Article 29 (chunk_id: CELEX_32022R2554_EN_TXT__block_28__chunk_1). Therefore, I cannot provide the details of Article 29 itself.
['CELEX_32022R2554_EN_TXT__block_16__chunk_0', 'CELEX_32022R2554_EN_TXT__block_16__chunk_1', 'CELEX_32022R2554_EN_TXT__block_28__chunk_0', 'CELEX_32022R2554_EN_TXT__block_28__chunk_1']
[{'chunk_id': 'CELEX_32022R2554_EN_TXT__block_16__chunk_0', 'doc_id': 'CELEX_32022R2554_EN_TXT', 'doc_title': 'REGULATION (EU) 2022/2554 OF THE EUROPEAN PARLIAMENT AND OF THE COUNCIL', 'doc_date': '14 December 2022', 'doc_subtitle': 'on digital operational resilience for the financial sector and amending Regulations (EC) No 

In [46]:
res

{'question': 'What does Article 29 say about ICT concentration risk?',
 'answer': 'The retrieved legal context does not provide the specific content of Article 29 regarding ICT concentration risk. It only mentions that financial entities must identify and assess all relevant risks in relation to contractual arrangements, including the possibility that such arrangements may contribute to reinforcing ICT concentration risk as referred to in Article 29 (chunk_id: CELEX_32022R2554_EN_TXT__block_28__chunk_1). Therefore, I cannot provide the details of Article 29 itself.',
 'chunk_ids': ['CELEX_32022R2554_EN_TXT__block_16__chunk_0',
  'CELEX_32022R2554_EN_TXT__block_16__chunk_1',
  'CELEX_32022R2554_EN_TXT__block_28__chunk_0',
  'CELEX_32022R2554_EN_TXT__block_28__chunk_1'],
 'sources': [{'chunk_id': 'CELEX_32022R2554_EN_TXT__block_16__chunk_0',
   'doc_id': 'CELEX_32022R2554_EN_TXT',
   'doc_title': 'REGULATION (EU) 2022/2554 OF THE EUROPEAN PARLIAMENT AND OF THE COUNCIL',
   'doc_date': '1

In [43]:
chat_history = [
    {"role": "user", "content": "What is DORA?"},
    {"role": "assistant", "content": "DORA is ..."}
]

In [44]:
chat_history

[{'role': 'user', 'content': 'What is DORA?'},
 {'role': 'assistant', 'content': 'DORA is ...'}]

In [1]:
from app.main import answer_question

res = answer_question(
    "What does Article 29 say about ICT concentration risk?",
    chat_history=[],
    return_context=True
)

print(res["rewritten_query"])
print(res["answer"])
print(res["chunk_ids"][:5])

What does Article 29 say about ICT concentration risk?
The retrieved legal context does not provide specific details about Article 29 or its contents regarding ICT concentration risk. Therefore, I cannot provide information about what Article 29 says.
['CELEX_32022R2554_EN_TXT__block_16__chunk_0', 'CELEX_32022R2554_EN_TXT__block_16__chunk_1', 'CELEX_32022R2554_EN_TXT__block_28__chunk_0', 'CELEX_32022R2554_EN_TXT__block_28__chunk_1']
